# Analyse des évaluations clients — secteur dermocosmétique

Projet de synthèse INF4173 — Université du Québec en Outaouais — Avril 2026
Auteure : Naima Ait Belkacem — Superviseure : Pre Hajar Moudoud

Ce carnet couvre la chaîne complète : chargement, contrôle qualité, prétraitement,
analyse exploratoire, analyses croisées, modélisation, puis export vers Power BI.

**Données requises** (non redistribuées ici, à télécharger depuis Kaggle) :
- `sephora_website_dataset.csv` — 9 168 produits
- `reviews_0-250.csv` … `reviews_1250-end.csv` — 1 094 411 avis

Adapter la variable `RACINE` ci-dessous à l'emplacement local des fichiers.

## 0. Configuration des chemins

Un seul endroit à modifier pour rejouer le carnet sur une autre machine.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ---- Seul bloc à adapter selon la machine ----------------------------------
RACINE   = Path(r"C:\uqo\s5\projet synthèse")
DATA     = RACINE / "dataset"
WEBSITE  = DATA / "sephora_website_dataset.csv" / "sephora_website_dataset.csv"
ARCHIVE  = DATA / "archive"
EXPORT   = RACINE / "powerbi"
# ---------------------------------------------------------------------------

EXPORT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 120)

for chemin in (WEBSITE, ARCHIVE):
    print(("OK   " if chemin.exists() else "MANQUANT "), chemin)

## 1. Chargement des données

In [ ]:
manquants = [c for c in (WEBSITE, ARCHIVE) if not c.exists()]
if manquants:
    raise FileNotFoundError("Introuvable : " + ", ".join(str(c) for c in manquants))

df_website = pd.read_csv(WEBSITE, low_memory=False)
df_website.columns = df_website.columns.str.strip()

print("Produits :", df_website.shape)
df_website.dtypes

In [ ]:
fichiers_avis = ["reviews_0-250.csv", "reviews_250-500.csv", "reviews_500-750.csv",
                 "reviews_750-1250.csv", "reviews_1250-end.csv"]

df_reviews = pd.concat(
    [pd.read_csv(ARCHIVE / f, low_memory=False) for f in fichiers_avis],
    ignore_index=True
)
df_reviews.columns = df_reviews.columns.str.strip()

ATTENDU = 1_094_411   # valeur citée dans le rapport
print("Avis :", df_reviews.shape)
print("Conforme au rapport :", len(df_reviews) == ATTENDU)
df_reviews.dtypes

## 2. Contrôle qualité

### 2.1 Valeurs manquantes

Contrôle des deux tables séparément. Placer deux instructions dans une même
cellule n'affiche que le résultat de la dernière : le premier contrôle passerait
inaperçu.

In [ ]:
print("=== Valeurs manquantes — Avis ===")
manquants_reviews = df_reviews.isnull().sum()
print(manquants_reviews[manquants_reviews > 0] if manquants_reviews.any()
      else "Aucune valeur manquante.")

print("\n=== Valeurs manquantes — Produits ===")
manquants_website = df_website.isnull().sum()
print(manquants_website[manquants_website > 0] if manquants_website.any()
      else "Aucune valeur manquante.")

`is_recommended` est de type `float64` : dans pandas, une colonne binaire sans
valeur manquante serait de type `int64`. Le type flottant indique donc la présence
de valeurs nulles. Ces avis sont ceux où la personne a noté le produit sans
répondre à la question de la recommandation.

Conséquence à retenir pour la suite : **le taux de recommandation doit être
calculé sur les avis où la question a reçu une réponse**, et non sur le total des
avis, sous peine de sous-estimer systématiquement le taux.

In [ ]:
n_total = len(df_reviews)
n_reponse = df_reviews["is_recommended"].notna().sum()

print(f"Avis au total            : {n_total:,}".replace(",", " "))
print(f"Avis avec réponse        : {n_reponse:,}".replace(",", " "))
print(f"Avis sans réponse        : {n_total - n_reponse:,}".replace(",", " "))
print(f"Part sans réponse        : {(n_total - n_reponse) / n_total:.1%}")

print("\nValeurs présentes :")
print(df_reviews["is_recommended"].value_counts(dropna=False))

### 2.2 Doublons et colonne technique

In [ ]:
print("Doublons — avis     :", df_reviews.duplicated().sum())
print("Doublons — produits :", df_website.duplicated().sum())

if "Unnamed: 0" in df_reviews.columns:
    df_reviews = df_reviews.drop(columns=["Unnamed: 0"])
    print("\nColonne technique 'Unnamed: 0' supprimée.")

df_reviews.columns.tolist()

### 2.3 Compatibilité des identifiants produits

Contrôle de la possibilité de joindre les deux tables.

In [ ]:
df_reviews["product_id"] = df_reviews["product_id"].astype(str)
df_website["id"] = df_website["id"].astype(str)

correspondances = df_reviews["product_id"].isin(df_website["id"]).sum()
print(f"Avis rattachables à un produit : {correspondances} sur {len(df_reviews)}")
print("\nFormat des identifiants — avis     :", df_reviews["product_id"].head(3).tolist())
print("Format des identifiants — produits :", df_website["id"].head(3).tolist())

Les identifiants sont incompatibles : aucune jointure directe n'est possible.
Les deux tables sont donc analysées séparément et rapprochées au niveau de la
marque et de la catégorie. Cette contrainte est documentée et non contournée.

## 3. Prétraitement

### 3.1 Conversion des dates et variables temporelles

In [ ]:
df_reviews["submission_time"] = pd.to_datetime(df_reviews["submission_time"])
df_reviews["annee"] = df_reviews["submission_time"].dt.year
df_reviews["mois"] = df_reviews["submission_time"].dt.month

df_reviews[["submission_time", "annee", "mois"]].head()

### 3.2 Normalisation des colonnes de texte

Les espaces invisibles et les chaines vides creent, dans Power BI, des categories
qui semblent identiques mais sont comptees separement. Les valeurs absentes
recoivent une etiquette explicite : c'est ce qui fait disparaitre les barres sans
nom observees sur les pages 3 et 4 du tableau de bord.

In [ ]:
for colonne in ["skin_type", "brand_name", "product_name"]:
    df_reviews[colonne] = (df_reviews[colonne].astype("string").str.strip()
                           .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA}))

for colonne in ["brand", "category", "name"]:
    df_website[colonne] = (df_website[colonne].astype("string").str.strip()
                           .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA}))

# Etiquette explicite plutot qu'une barre vide dans Power BI
df_reviews["skin_type"] = df_reviews["skin_type"].fillna("non declare")

print("Types de peau presents :")
print(df_reviews["skin_type"].value_counts(dropna=False))

### 3.3 Construction des indicateurs dérivés

In [ ]:
def classer_prix(prix):
    """Segmente un produit en trois gammes tarifaires."""
    if prix <= 25:
        return "entrée de gamme"
    elif prix <= 75:
        return "milieu de gamme"
    return "haut de gamme"


def classer_note(note):
    """Traduit une note 1-5 en polarité (approximation avant analyse NLP)."""
    if note <= 2:
        return "négatif"
    elif note == 3:
        return "neutre"
    return "positif"


df_website["gamme_prix"] = df_website["price"].apply(classer_prix)
df_reviews["sentiment_label"] = df_reviews["rating"].apply(classer_note)

# Gamme de prix côté avis, à partir du prix porté par chaque avis
df_reviews["gamme_prix"] = df_reviews["price_usd"].apply(classer_prix)

print("Gammes de prix (produits) :")
print(df_website["gamme_prix"].value_counts())
print("\nPolarité (avis) :")
print(df_reviews["sentiment_label"].value_counts())

### 3.4 Structure finale des deux tables

In [ ]:
print("=== Avis ===")
print(f"Dimensions : {df_reviews.shape}")
print(f"Colonnes   : {df_reviews.columns.tolist()}")

print("\n=== Produits ===")
print(f"Dimensions : {df_website.shape}")
print(f"Colonnes   : {df_website.columns.tolist()}")

## 4. Analyse exploratoire

### 4.1 Distribution des notes

In [ ]:
print(df_reviews["rating"].describe())
print("\nRépartition :")
print(df_reviews["rating"].value_counts().sort_index())

df_reviews["rating"].hist(bins=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5], color="steelblue",
                          edgecolor="white")
plt.title("Distribution des notes")
plt.xlabel("Note")
plt.ylabel("Nombre d'avis")
plt.tight_layout()
plt.show()

### 4.2 Volume d'avis par marque

In [ ]:
df_reviews["brand_name"].value_counts().head(10).plot(kind="bar")
plt.title("Top 10 marques par nombre d'avis")
plt.xlabel("Marque")
plt.ylabel("Nombre d'avis")
plt.show()

### 4.3 Évolution temporelle

In [ ]:
# Volume d'avis par année
avis_par_annee = df_reviews.groupby("annee")["rating"].count()

plt.figure(figsize=(10, 5))
avis_par_annee.plot(kind="bar", color="steelblue")
plt.title("Évolution du volume d'avis par année")
plt.xlabel("Année")
plt.ylabel("Nombre d'avis")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(avis_par_annee)

# Note moyenne par année
note_par_annee = df_reviews.groupby("annee")["rating"].mean()

plt.figure(figsize=(10, 5))
note_par_annee.plot(kind="line", marker="o", color="steelblue")
plt.title("Évolution de la note moyenne par année")
plt.xlabel("Année")
plt.ylabel("Note moyenne")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(note_par_annee)

### 4.4 Note et comportement de recommandation

In [ ]:
print("Corrélation note / recommandation :")
print(df_reviews[["rating", "is_recommended"]].corr())

note_par_reco = df_reviews.groupby("is_recommended")["rating"].mean()
print("\nNote moyenne selon la recommandation :")
print(note_par_reco)

note_par_reco.plot(kind="bar", color="steelblue")
plt.title("Note moyenne selon la recommandation")
plt.xlabel("Recommandation (0 = Non, 1 = Oui)")
plt.ylabel("Note moyenne")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 4.5 Analyse de sentiment (NLP — VADER)

Les pourcentages du graphique sont recalculés à partir du résultat de l'analyse,
jamais saisis en dur : le graphique ne peut donc pas diverger des données si
l'échantillon change.

In [ ]:
# Installation si nécessaire : pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

# Échantillon de 10 000 avis — l'analyse du corpus complet serait très longue
echantillon = df_reviews["review_text"].dropna().sample(10000, random_state=42)
scores = echantillon.apply(lambda t: analyzer.polarity_scores(t)["compound"])


def classer_vader(score):
    if score >= 0.05:
        return "positif"
    elif score <= -0.05:
        return "négatif"
    return "neutre"


sentiment_vader = scores.apply(classer_vader)

repartition = (sentiment_vader.value_counts(normalize=True)
               .mul(100).round(1)
               .reindex(["positif", "négatif", "neutre"]))

print("Distribution du sentiment VADER (%) :")
print(repartition)

couleurs = {"positif": "#5cb85c", "négatif": "#d9534f", "neutre": "#f0ad4e"}

plt.figure(figsize=(8, 5))
barres = plt.bar(repartition.index, repartition.values,
                 color=[couleurs[i] for i in repartition.index])
plt.title("Distribution des sentiments — analyse VADER (NLP)")
plt.ylabel("Pourcentage (%)")
plt.ylim(0, 100)

for barre, valeur in zip(barres, repartition.values):
    plt.text(barre.get_x() + barre.get_width() / 2, barre.get_height() + 1,
             f"{valeur}%", ha="center", fontsize=12)

plt.tight_layout()
plt.show()

### 4.6 Prix et satisfaction

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df_website["price"], df_website["rating"], alpha=0.3, color="steelblue")
axes[0].set_xlabel("Prix ($)")
axes[0].set_ylabel("Note moyenne")
axes[0].set_title("Relation entre le prix et la note")

ordre_gammes = ["entrée de gamme", "milieu de gamme", "haut de gamme"]
note_gamme = df_website.groupby("gamme_prix")["rating"].mean().reindex(ordre_gammes)

note_gamme.plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("Note moyenne par gamme de prix")
axes[1].set_xlabel("Gamme de prix")
axes[1].set_ylabel("Note moyenne")
# Axe volontairement tronqué pour rendre l'écart lisible.
# Cet écart reste de 0,17 point sur une échelle de 5 : il est négligeable.
axes[1].set_ylim(3.5, 4.5)
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

print("Corrélation prix / note :")
print(df_website[["price", "rating"]].corr())
print("\nNote moyenne par gamme :")
print(note_gamme.round(2))

### 4.7 Catalogue produits : marques, prix, catégories

In [ ]:
top_brands = df_website["brand"].value_counts().head(10)

top_brands.plot(kind="bar")
plt.title("Top 10 marques par nombre de produits")
plt.xlabel("Marque")
plt.ylabel("Nombre de produits")
plt.show()

In [ ]:
import seaborn as sns

sns.boxplot(x=df_website["price"])
plt.title("Distribution des prix")
plt.show()

In [ ]:
# Note moyenne par catégorie de produit
note_par_categorie = df_website.groupby("category")["rating"].mean().sort_values(ascending=False).head(15)

plt.figure(figsize=(12, 6))
note_par_categorie.plot(kind="bar", color="steelblue")
plt.title("Note moyenne par catégorie de produit")
plt.xlabel("Catégorie")
plt.ylabel("Note moyenne")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(note_par_categorie)

## 5. Analyses croisées

### 5.1 Popularité et qualité perçue

In [ ]:
# 5.1 Note moyenne par marque — Top 15 (dataset Reviews)
note_par_marque = df_reviews.groupby("brand_name")["rating"].agg(
    note_moyenne="mean",
    nb_avis="count"
).reset_index()

# Filtrer les marques avec au moins 1000 avis
note_par_marque_filtre = note_par_marque[
    note_par_marque["nb_avis"] >= 1000
].sort_values("note_moyenne", ascending=False).head(15)

print("Top 15 marques par note moyenne (min. 1000 avis) :")
print(note_par_marque_filtre.to_string(index=False))

# Visualisation
plt.figure(figsize=(12, 6))
plt.barh(note_par_marque_filtre["brand_name"],
         note_par_marque_filtre["note_moyenne"],
         color="steelblue")
plt.xlabel("Note moyenne")
plt.title("Top 15 marques par note moyenne (min. 1000 avis)")
plt.xlim(3.5, 5.0)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Comparaison : volume d'avis vs note moyenne
# Les marques populaires sont-elles aussi les mieux notées ?

top_volume = df_reviews.groupby("brand_name")["rating"].agg(
    note_moyenne="mean",
    nb_avis="count"
).reset_index().sort_values("nb_avis", ascending=False).head(10)

print("Top 10 marques par volume d'avis :")
print(top_volume[["brand_name", "nb_avis", "note_moyenne"]].to_string(index=False))

plt.figure(figsize=(10, 6))
plt.scatter(top_volume["nb_avis"],
            top_volume["note_moyenne"],
            color="steelblue", s=100)

for _, row in top_volume.iterrows():
    plt.annotate(row["brand_name"],
                (row["nb_avis"], row["note_moyenne"]),
                textcoords="offset points",
                xytext=(5, 5), fontsize=8)

plt.xlabel("Nombre d'avis")
plt.ylabel("Note moyenne")
plt.title("Volume d'avis vs Note moyenne — Top 10 marques")
plt.tight_layout()
plt.show()

### 5.2 Gamme de prix côté avis

In [ ]:
note_par_gamme = df_reviews.groupby("gamme_prix")["rating"].agg(
    note_moyenne="mean", nb_avis="count").reindex(ordre_gammes)

print("Note moyenne par gamme de prix (avis) :")
print(note_par_gamme.round(3))

note_par_gamme["note_moyenne"].plot(kind="bar", color="steelblue", figsize=(8, 5))
plt.title("Note moyenne par gamme de prix")
plt.xlabel("Gamme de prix")
plt.ylabel("Note moyenne")
plt.xticks(rotation=0)
plt.ylim(3.8, 4.5)  # axe tronqué — écart réel inférieur à 0,2 point
plt.tight_layout()
plt.show()

### 5.3 Marque et type de peau

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Top marques sélectionnées
top_marques = ["CLINIQUE", "Tatcha", 
               "Drunk Elephant", "fresh", 
               "Dermalogica", "The Ordinary",
               "Origins", "LANEIGE"]

df_cross = df_reviews[
    df_reviews["brand_name"].isin(top_marques)
]

# Créer le tableau croisé
pivot = df_cross.groupby(
    ["brand_name", "skin_type"]
)["rating"].mean().unstack()

print("Note moyenne par marque et type de peau :")
print(pivot.round(2))

# Heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(pivot, 
            annot=True, 
            fmt=".2f",
            cmap="YlOrRd", 
            vmin=3.8, vmax=4.8,
            linewidths=0.5)
plt.title("Note moyenne par marque et type de peau")
plt.xlabel("Type de peau")
plt.ylabel("Marque")
plt.tight_layout()
plt.show()

### 5.4 Les paradoxes de la recommandation

In [ ]:
# 5.3 Paradoxe de la recommandation

# Cas 1 : clients satisfaits qui ne recommandent pas
paradoxe1 = df_reviews[
    (df_reviews["rating"] >= 4) & 
    (df_reviews["is_recommended"] == 0)
]

# Cas 2 : clients insatisfaits qui recommandent quand même
paradoxe2 = df_reviews[
    (df_reviews["rating"] <= 2) & 
    (df_reviews["is_recommended"] == 1)
]

total = len(df_reviews)

print("=== PARADOXE DE LA RECOMMANDATION ===")
print(f"\nCas 1 — Satisfaits mais ne recommandent pas :")
print(f"Nombre : {len(paradoxe1)}")
print(f"Pourcentage : {len(paradoxe1)/total*100:.1f}%")

print(f"\nCas 2 — Insatisfaits mais recommandent quand même :")
print(f"Nombre : {len(paradoxe2)}")
print(f"Pourcentage : {len(paradoxe2)/total*100:.1f}%")

# Quelles marques génèrent le plus de Cas 1 ?
print("\nTop 10 marques — Cas 1 (satisfaits mais non-recommandants) :")
print(paradoxe1["brand_name"].value_counts().head(10))

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

paradoxe1["brand_name"].value_counts().head(8).plot(
    kind="bar", ax=axes[0], color="#f0ad4e",
    title="Satisfaits mais ne recommandent pas\n(Top 8 marques)")
axes[0].set_xlabel("Marque")
axes[0].set_ylabel("Nombre d'avis")
axes[0].tick_params(axis="x", rotation=45)

paradoxe2["brand_name"].value_counts().head(8).plot(
    kind="bar", ax=axes[1], color="#5cb85c",
    title="Insatisfaits mais recommandent quand même\n(Top 8 marques)")
axes[1].set_xlabel("Marque")
axes[1].set_ylabel("Nombre d'avis")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Marques paradoxales identifiées en 5.3
marques_paradoxe = ["Tatcha", "Drunk Elephant", 
                     "The Ordinary", "fresh",
                     "Youth To The People", "CLINIQUE",
                     "LANEIGE", "Glow Recipe"]

# Cas 1 — Satisfaits mais ne recommandent pas
paradoxe1 = df_reviews[
    (df_reviews["rating"] >= 4) & 
    (df_reviews["is_recommended"] == 0) &
    (df_reviews["brand_name"].isin(marques_paradoxe))
]

# Cas 2 — Insatisfaits mais recommandent
paradoxe2 = df_reviews[
    (df_reviews["rating"] <= 2) & 
    (df_reviews["is_recommended"] == 1) &
    (df_reviews["brand_name"].isin(marques_paradoxe))
]

# Top produits dans chaque cas
print("=== CAS 1 — Satisfaits mais ne recommandent pas ===")
print("Top 10 produits :")
print(paradoxe1.groupby(
    ["brand_name", "product_name"])
    .size().sort_values(ascending=False)
    .head(10).to_string())

print("\n=== CAS 2 — Insatisfaits mais recommandent ===")
print("Top 10 produits :")
print(paradoxe2.groupby(
    ["brand_name", "product_name"])
    .size().sort_values(ascending=False)
    .head(10).to_string())

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# === CAS 1 ===
top_cas1 = paradoxe1.groupby(
    ["brand_name", "product_name"]
).size().sort_values(ascending=False).head(8)

labels1 = [
    f"{b} - {p[:40]}..." if len(p) > 40 else f"{b} - {p}"
    for b, p in top_cas1.index
]

axes[0].bar(range(len(top_cas1)), top_cas1.values, color="#f0ad4e")

axes[0].set_title("Cas 1 — Produits : Satisfaits mais ne recommandent pas")
axes[0].set_ylabel("Nombre d'avis")
axes[0].set_xticks(range(len(labels1)))
axes[0].set_xticklabels(labels1, rotation=30, ha='right')

# === CAS 2 ===
top_cas2 = paradoxe2.groupby(
    ["brand_name", "product_name"]
).size().sort_values(ascending=False).head(8)

labels2 = [
    f"{b} - {p[:40]}..." if len(p) > 40 else f"{b} - {p}"
    for b, p in top_cas2.index
]

axes[1].bar(range(len(top_cas2)), top_cas2.values, color="#5cb85c")

axes[1].set_title("Cas 2 — Produits : Insatisfaits mais recommandent")
axes[1].set_ylabel("Nombre d'avis")
axes[1].set_xticks(range(len(labels2)))
axes[1].set_xticklabels(labels2, rotation=30, ha='right')

plt.tight_layout()
plt.show()

## 6. Modélisation prédictive

Un arbre de décision est entraîné pour classer l'importance des variables dans le
comportement de recommandation. Les libellés du graphique sont dérivés
directement de `X.columns` : ils ne peuvent pas se désaligner des variables
réellement utilisées si l'ordre des colonnes change.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

df_ml = df_reviews[["skin_type", "price_usd", "rating",
                    "is_recommended", "brand_name"]].dropna().copy()

df_ml["skin_type_enc"] = LabelEncoder().fit_transform(df_ml["skin_type"])
df_ml["brand_enc"] = LabelEncoder().fit_transform(df_ml["brand_name"])

variables = ["skin_type_enc", "price_usd", "rating", "brand_enc"]
libelles = {"skin_type_enc": "Type de peau", "price_usd": "Prix",
            "rating": "Note", "brand_enc": "Marque"}

X = df_ml[variables]
y = df_ml["is_recommended"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

modele = DecisionTreeClassifier(max_depth=4, random_state=42)
modele.fit(X_train, y_train)

print("=== Rapport de classification ===")
print(classification_report(y_test, modele.predict(X_test)))

importances = (pd.DataFrame({"variable": [libelles[c] for c in X.columns],
                             "importance": modele.feature_importances_})
               .sort_values("importance", ascending=False))

print("=== Importance des variables ===")
print(importances.to_string(index=False))

plt.figure(figsize=(8, 5))
plt.barh(importances["variable"], importances["importance"], color="steelblue")
plt.title("Variables déterminantes dans la prédiction de recommandation")
plt.xlabel("Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Lecture critique du modèle

La note concentre la quasi-totalité de l'importance. Ce n'est pas un résultat
d'analyse : c'est une **fuite de cible**. La note et la recommandation mesurent la
même satisfaction sous-jacente, le modèle ne fait donc que reformuler une
tautologie.

Le contrôle ci-dessous entraîne le même arbre **sans la note**. C'est cette
version qui répond à une question utile : les caractéristiques du produit et le
profil de la personne permettent-ils, à eux seuls, de prédire la recommandation ?

In [ ]:
variables_sans_note = ["skin_type_enc", "price_usd", "brand_enc"]

X2 = df_ml[variables_sans_note]
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.2, random_state=42, stratify=y)

modele2 = DecisionTreeClassifier(max_depth=4, random_state=42)
modele2.fit(X2_train, y2_train)

print("=== Sans la variable Note ===")
print(classification_report(y2_test, modele2.predict(X2_test)))

importances2 = (pd.DataFrame({"variable": [libelles[c] for c in X2.columns],
                              "importance": modele2.feature_importances_})
                .sort_values("importance", ascending=False))
print(importances2.to_string(index=False))

print("\nRéférence — part de la classe majoritaire :", round(y.mean(), 4))

## 7. Export vers Power BI

Les données sont propres dans pandas. Le risque se situe à l'écriture du CSV puis
à sa relecture par Power BI :

1. **Caractères de rupture.** Les noms de produits contiennent des retours à la
   ligne, des guillemets (les pouces : `5"`) et des virgules. Le lecteur CSV de
   Power BI coupe alors l'enregistrement au mauvais endroit et décale toutes les
   colonnes suivantes — une valeur de prix peut se retrouver dans la colonne
   catégorie.
2. **Séparateur décimal.** Le CSV est écrit avec le point. Sur un poste
   configuré en français, Power BI peut lire le point comme séparateur de
   milliers. Il faut donc importer en précisant la locale anglaise, ou fournir
   des colonnes que Power BI ne peut pas mal interpréter.
3. **Valeurs manquantes.** `is_recommended` contient des vides. Le taux de
   recommandation doit s'appuyer sur une colonne où le vide reste un vide, pour
   que Power BI l'exclue du calcul au lieu de le compter comme un refus.

Les trois points sont traités ici, à la source.

In [ ]:
import csv
import re


def assainir(serie):
    """Neutralise les caractères qui font dérailler un lecteur CSV."""
    return (serie.astype(str)
            .str.replace(r"[\r\n\t]+", " ", regex=True)   # ruptures de ligne
            .str.replace('"', "", regex=False)              # guillemets (pouces)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip())


# ---- Table des avis --------------------------------------------------------
reviews_export = df_reviews[[
    "rating", "is_recommended", "skin_type", "brand_name",
    "price_usd", "annee", "mois", "sentiment_label", "product_name"
]].copy()

# Entier nullable : 1, 0, ou vide. Le vide reste vide et sera ignoré par Power BI.
reviews_export["is_recommended"] = (
    pd.to_numeric(reviews_export["is_recommended"], errors="coerce").astype("Int64"))

for colonne in ["skin_type", "brand_name", "sentiment_label", "product_name"]:
    reviews_export[colonne] = assainir(reviews_export[colonne])

# ---- Table des produits ----------------------------------------------------
website_export = df_website[[
    "brand", "category", "name", "price", "rating",
    "number_of_reviews", "online_only", "exclusive",
    "limited_edition", "gamme_prix"
]].copy()

for colonne in ["brand", "category", "name", "gamme_prix"]:
    website_export[colonne] = assainir(website_export[colonne])

for colonne in ["price", "rating"]:
    website_export[colonne] = pd.to_numeric(website_export[colonne], errors="coerce")

# ---- Écriture --------------------------------------------------------------
# ---- Garde-fous : le script s'arrete plutot que d'exporter des donnees fausses
assert reviews_export["rating"].between(1, 5).all(), "Note hors intervalle 1-5"
assert set(reviews_export["is_recommended"].dropna().unique()) <= {0, 1}, \
    "is_recommended contient autre chose que 0 et 1"
assert website_export["rating"].dropna().between(0, 5).all(), "Note produit hors 0-5"
assert not website_export["category"].str.fullmatch(
    r"\d+([.,]\d+)?", na=False).any(), "Categorie numerique : colonnes decalees"
assert len(reviews_export) == ATTENDU, \
    f"{len(reviews_export)} lignes exportees au lieu de {ATTENDU}"

# lineterminator evite les fins de ligne Windows, mal gerees par certains lecteurs
options = dict(index=False, encoding="utf-8-sig", sep=",", decimal=".",
               quoting=csv.QUOTE_ALL, lineterminator="\n")

reviews_export.to_csv(EXPORT / "reviews_clean.csv", **options)
website_export.to_csv(EXPORT / "website_clean.csv", **options)

print("Avis exportés     :", reviews_export.shape)
print("Produits exportés :", website_export.shape)
print("Dossier           :", EXPORT)

### 7.1 Vérification par relecture

Écrire un fichier ne prouve pas qu'il est lisible. Les fichiers sont donc relus
depuis le disque et comparés à ce qui devait être écrit. Si ce contrôle passe et
que Power BI affiche malgré tout des valeurs incohérentes, le problème est du
côté du paramétrage de l'import dans Power BI, pas des données.

In [ ]:
controles = []

for nom, attendu in [("reviews_clean.csv", reviews_export),
                     ("website_clean.csv", website_export)]:
    relu = pd.read_csv(EXPORT / nom, encoding="utf-8-sig")

    controles.append((f"{nom} — nombre de lignes",
                      len(relu) == len(attendu), f"{len(relu)} / {len(attendu)}"))
    controles.append((f"{nom} — nombre de colonnes",
                      relu.shape[1] == attendu.shape[1],
                      f"{relu.shape[1]} / {attendu.shape[1]}"))

    numeriques = [c for c in ["rating", "price", "price_usd", "number_of_reviews"]
                  if c in relu.columns]
    for c in numeriques:
        ok = pd.api.types.is_numeric_dtype(relu[c])
        controles.append((f"{nom} — '{c}' numérique", ok, str(relu[c].dtype)))

    if "category" in relu.columns:
        intrus = relu["category"].astype(str).str.match(r"^\d+([.,]\d+)?$", na=False).sum()
        controles.append((f"{nom} — catégories non numériques", intrus == 0,
                          f"{intrus} intrus"))

    if "rating" in relu.columns:
        borne = relu["rating"].between(0, 5).all()
        controles.append((f"{nom} — notes dans [0 ; 5]", borne,
                          f"min {relu['rating'].min()} / max {relu['rating'].max()}"))

print(f"{'CONTRÔLE':<45} {'ÉTAT':<8} DÉTAIL")
print("-" * 78)
for libelle, ok, detail in controles:
    print(f"{libelle:<45} {'OK' if ok else 'ÉCHEC':<8} {detail}")

print("\n>>> TOUS LES CONTRÔLES PASSENT" if all(c[1] for c in controles)
      else "\n>>> AU MOINS UN CONTRÔLE A ÉCHOUÉ")

### 7.2 Valeurs de référence pour Power BI

Ces chiffres sont calculés ici, dans pandas, à partir des données propres. Ils
servent de référence : après import, chaque visuel de Power BI doit afficher
exactement ces valeurs. Tout écart signale une erreur d'import ou de mesure DAX,
pas une erreur d'analyse.

In [ ]:
print("=== VALEURS ATTENDUES DANS POWER BI ===\n")

print("Page 1 — Vue générale")
print(f"  Note moyenne        : {df_reviews['rating'].mean():.2f}")
print(f"  Nombre d'avis       : {len(df_reviews):,}".replace(",", " "))
print(f"  Nombre de marques   : {df_reviews['brand_name'].nunique()}")

print("\nPage 3 — Profil client")
profil = df_reviews.groupby("skin_type").agg(
    note_moyenne=("rating", "mean"),
    taux_recommandation=("is_recommended", "mean"),
    nb_avis=("rating", "count"))
print(profil.round(3).to_string())

print("\nPage 4 — Analyse produit")
print("  Note moyenne par gamme de prix :")
print(df_website.groupby("gamme_prix")["rating"].mean()
      .reindex(ordre_gammes).round(2).to_string())
print("\n  Top 5 produits par nombre d'avis :")
print(df_website.nlargest(5, "number_of_reviews")[["name", "number_of_reviews"]]
      .to_string(index=False))

## 8. Mesures DAX à utiliser dans Power BI

Une fois les fichiers ci-dessus importés, remplacer les mesures existantes par
celles-ci.

```dax
Note_Moyenne_Marque = AVERAGE(reviews_clean[rating])

Taux_Recommandation = AVERAGE(reviews_clean[is_recommended])

Note_Moyenne_Produit = AVERAGE(website_clean[rating])

Nombre_Avis = COUNTROWS(reviews_clean)

Avis_Produit = SUM(website_clean[number_of_reviews])
```

**Pourquoi ces formulations.**

`Taux_Recommandation` s'appuie sur `AVERAGE` d'une colonne qui ne contient que 1,
0 ou du vide. La moyenne d'une colonne binaire *est* le taux, et `AVERAGE` ignore
les vides : le taux se calcule donc sur les avis où la question a reçu une
réponse. La version précédente comparait la colonne au texte `"1.0"` et divisait
par le total des avis, ce qui écrasait le résultat dès qu'un segment comportait
beaucoup de non-réponses.

`Note_Moyenne_Produit` n'a plus besoin de `SUBSTITUTE` ni de `VALUE` : la colonne
arrive déjà numérique et la conversion en DAX, sensible aux paramètres régionaux,
disparaît avec le problème qu'elle tentait de contourner.

`Avis_Produit` remplace le comptage de lignes dans le visuel « Top 10 produits ».
La table des produits contient une ligne par produit : compter les lignes donne 1
partout. C'est `number_of_reviews` qu'il faut additionner.

**Import dans Power BI** : *Obtenir les données → Texte/CSV*, puis dans la boîte
de dialogue, régler **Origine du fichier** sur `65001: Unicode (UTF-8)` et
**Paramètres régionaux** sur `Anglais (États-Unis)`. Ce dernier réglage garantit
que le point est lu comme séparateur décimal. Vérifier ensuite dans Power Query
que `rating`, `price`, `price_usd` et `number_of_reviews` portent bien le type
Nombre décimal, et `is_recommended` le type Nombre entier.